## Setup

Imports, configuration constants, and helper functions used throughout the notebook — split into small, single-purpose blocks below.

### Imports

Standard library, third-party, and scikit-learn imports.

In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import re
import warnings

import librosa
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Audio, HTML, display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import umap

### Display and Plotting Defaults

Warning filters and pandas/plotly display settings.

In [2]:
# Display and plotting defaults
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
px.defaults.template = "plotly_white"

### Data Location

Resolves `DATA_DIR` so the notebook runs the same way whether it's opened from `notebooks/` or the repo root.

In [3]:
# Data location: resolve DATA_DIR so this notebook runs the same way from notebooks/ or the repo root
REPO_ROOT = Path.cwd()
DATA_CANDIDATES = [
    REPO_ROOT / "data" / "raw" / "CatSound_originals",
    REPO_ROOT.parent / "data" / "raw" / "CatSound_originals",
    Path("../data/raw/CatSound_originals"),
]
DATA_DIR = next((candidate for candidate in DATA_CANDIDATES if candidate.exists()), DATA_CANDIDATES[-1])
AUDIO_EXTENSIONS = {".mp3", ".wav", ".flac", ".m4a", ".ogg"}

### Audio Ingestion Contract

`SAMPLE_RATE` and `CLIP_DURATION_S` control every audio load and the fixed-length crop/pad applied before feature extraction — change either constant and rerun to try a different ingestion contract. `RANDOM_SEED` keeps sampling and models reproducible.

In [4]:
# Audio ingestion contract: adjust these two to change every load/feature cell below
SAMPLE_RATE = 44100    # Hz, used for every librosa.load call in this notebook
CLIP_DURATION_S = 2.0  # seconds, center-crop/pad target (matches the conv-7 preprocessing in .keep/EDA/cjk-audio-conv-7.ipynb)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

### Helper Functions: File Identity

`group_key()`, `md5_of_file()`, and `is_audio_file()` support duplicate and segment-group detection during dataset inventory.

In [5]:
# Helper functions: file identity and inventory
def group_key(filename: str) -> str:
    """Return a normalized key for files that belong to the same recording group."""
    key = Path(filename).stem
    key = re.sub(r"\s*\(\d+\)$", "", key)
    key = key.replace("_opt", "")
    return key.lower()


def md5_of_file(path: Path) -> str:
    """Return the MD5 digest of a file so exact duplicates can be detected."""
    return hashlib.md5(path.read_bytes()).hexdigest()


def is_audio_file(path: Path) -> bool:
    """Check whether a path has one of the audio extensions used in this dataset."""
    return path.suffix.lower() in AUDIO_EXTENSIONS

### Helper Functions: Audio Analysis

`audio_duration()` and `native_sample_rate()` read a clip's length and original sample rate from its header without decoding the whole file; `fit_to_duration()` applies the fixed-length crop/pad.

In [6]:
# Helper functions: audio analysis and preprocessing
def audio_duration(path: Path) -> float:
    """Return the audio duration in seconds using librosa's metadata reader."""
    return float(librosa.get_duration(path=str(path)))


def native_sample_rate(path: Path) -> int:
    """Return a file's original sample rate by reading its header, without decoding the audio."""
    return int(librosa.get_samplerate(str(path)))


def fit_to_duration(audio: np.ndarray, sample_rate: int, duration_s: float) -> np.ndarray:
    """Center-crop or zero-pad audio to an exact duration, controlled by CLIP_DURATION_S."""
    target_length = int(round(duration_s * sample_rate))
    current_length = len(audio)
    if current_length < target_length:
        total_pad = target_length - current_length
        pad_before = total_pad // 2
        pad_after = total_pad - pad_before
        return np.pad(audio, (pad_before, pad_after), mode="constant")
    if current_length > target_length:
        total_trim = current_length - target_length
        trim_before = total_trim // 2
        return audio[trim_before:trim_before + target_length]
    return audio

## Dataset At A Glance (computed directly from `data/raw/CatSound_originals`, 2953 files)

**Duration (seconds)**

| Stat | Value |
|---|---|
| Mean | 3.87 |
| Spread (std) | 1.78 |
| Range | 0.34 – 16.82 |
| Median | 3.79 |
| p95 | 6.82 |

**Native sample rate (Hz)** — read from each file's header before any resampling:

| Rate | Files | Share |
|---|---|---|
| 8000 | 1 | 0.0% |
| 11025 | 8 | 0.3% |
| 22050 | 5 | 0.2% |
| 32000 | 24 | 0.8% |
| 44100 | 2915 | 98.7% |

Mean = 43,862 Hz, spread (std) = 2,316 Hz — small relative to the mean because the dataset is almost entirely 44100 Hz with a handful of low-rate outliers.

**Optimal sample rate — there is no single answer, it depends on what you're optimizing for:**
- *Maximum fidelity, least information lost overall:* **44100 Hz** (the mode). Only the 38 outlier files (1.3%: the 8000/11025/22050/32000 Hz ones) need upsampling; the other 98.7% keep their full native detail untouched. This is the strongest choice if analysis quality is the priority — it is *not* a worse option just because it's a high number.
- *Compute, storage, and real-time latency:* a lower rate like 22050 Hz halves the sample count per file, which matters for a future record-and-predict API — but it downsamples the *dominant* 44100 Hz files, discarding real content above ~11 kHz for 98.7% of the dataset, not just a marginal few.
- *Never upsample a single file:* 8000 Hz (the true minimum) — the most conservative floor, but it throws away fidelity from nearly the whole dataset just to avoid touching one outlier file.
- Cat vocalizations are mostly low-frequency, but broadband elements like hisses and spits (Defence/Warning classes) can carry real energy above 10 kHz — so downsampling isn't free even for this dataset.

In [7]:
# Static snapshot chart — matches the table above, doesn't depend on df existing yet.
native_sr_snapshot = {8000: 1, 11025: 8, 22050: 5, 32000: 24, 44100: 2915}
mean_sr_snapshot = 43862.5
mode_sr_snapshot = 44100

glance_sr_fig = go.Figure()
glance_sr_fig.add_trace(go.Bar(
    x=list(native_sr_snapshot.keys()),
    y=list(native_sr_snapshot.values()),
    width=[1500] * len(native_sr_snapshot),
    text=[f"{count} files ({count / sum(native_sr_snapshot.values()) * 100:.1f}%)" for count in native_sr_snapshot.values()],
    textposition="outside",
    marker_color=["crimson" if sr == mode_sr_snapshot else "steelblue" for sr in native_sr_snapshot],
    name="Native sample rate",
))
glance_sr_fig.add_vline(
    x=mean_sr_snapshot, line_dash="dash", line_color="orange",
    annotation_text=f"mean ≈ {mean_sr_snapshot:,.0f} Hz", annotation_position="top left",
)
glance_sr_fig.add_vline(
    x=SAMPLE_RATE, line_dash="dot", line_color="green",
    annotation_text=f"current SAMPLE_RATE = {SAMPLE_RATE} Hz", annotation_position="bottom right",
)
glance_sr_fig.update_layout(
    title="Native sample rate distribution (static snapshot, log-scaled to show the minority bars)",
    xaxis_title="Native sample rate (Hz)",
    yaxis_title="File count",
    yaxis_type="log",
    showlegend=False,
)
display(HTML(glance_sr_fig.to_html(include_plotlyjs="cdn", full_html=False)))

# Kitty3000 Audio EDA and Production Fit

## Dataset Inventory

The cells below walk `DATA_DIR` once, list the 10 class subfolders, and record every file's label, filename, path, and size in `df`.

In [8]:
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

classes = sorted([item.name for item in DATA_DIR.iterdir() if item.is_dir()])
audio_files = []
for label in classes:
    for path in sorted((DATA_DIR / label).iterdir()):
        if path.is_file() and is_audio_file(path):
            audio_files.append({
                "label": label,
                "filename": path.name,
                "path": path,
                "file_size_kb": round(path.stat().st_size / 1024, 1),
            })

df = pd.DataFrame(audio_files)
df["group"] = df["filename"].map(group_key)

In [9]:
display(pd.DataFrame({"label": classes, "file_count": [int((df["label"] == label).sum()) for label in classes]}))
display(df.head())
print(f"{len(classes)} classes, {len(df)} audio files")

,label,file_count
0,Angry,300
1,Defence,291
2,Fighting,299
3,Happy,295
4,HuntingMind,289
5,Mating,300
6,MotherCall,296
7,Paining,287
8,Resting,296
9,Warning,300


,label,filename,path,file_size_kb,group
0,Angry,Cat_Angry1026.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,37.5,cat_angry1026
1,Angry,Cat_Angry1027.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,44.4,cat_angry1027
2,Angry,Cat_Angry1028.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,46.0,cat_angry1028
3,Angry,Cat_Angry1029.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,46.0,cat_angry1029
4,Angry,Cat_Angry1030.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,37.5,cat_angry1030


10 classes, 2953 audio files


## Production Fit Analysis

### Data Quality Checks

Duration and MD5 hashes are computed once, then reused to check for exact duplicate files and multi-part recording segments, before summarizing duration and file-size stats per class.

In [10]:
if "duration_s" not in df.columns:
    df["duration_s"] = [audio_duration(path) for path in df["path"]]

if "md5" not in df.columns:
    df["md5"] = [md5_of_file(path) for path in df["path"]]

In [11]:
md5_counts = df["md5"].value_counts()
exact_duplicates = df[df["md5"].isin(md5_counts[md5_counts > 1].index)].sort_values(["md5", "label", "filename"])

print(f"Exact duplicate files: {len(exact_duplicates)}")
display(exact_duplicates[["label", "filename", "md5"]].head(20))

Exact duplicate files: 0


,label,filename,md5


In [12]:
group_counts = df["group"].value_counts()
segment_groups = group_counts[group_counts > 1]

print(f"Segment groups with more than one file: {len(segment_groups)}")
print(f"Files inside repeated segment groups: {int(segment_groups.sum())}")
display(segment_groups.head(20))

Segment groups with more than one file: 149
Files inside repeated segment groups: 398


group
yashll_conf_cat0261fit    24
fight_confusion            9
cat02                      7
cat04                      7
cat05                      7
cat03                      6
cat06                      6
cat01                      6
cat07                      5
cat08                      5
cat010                     5
cat09                      4
cat13                      4
cat10                      4
online_cat0533veryang      3
cat014                     3
cat14                      3
lastentry_cat1363hiss      3
cat11                      3
yashll_cat0243fit          3
Name: count, dtype: int64

In [13]:
class_summary = (
    df.groupby("label")
    .agg(
        file_count=("filename", "size"),
        mean_duration_s=("duration_s", "mean"),
        median_duration_s=("duration_s", "median"),
        std_duration_s=("duration_s", "std"),
        min_duration_s=("duration_s", "min"),
        max_duration_s=("duration_s", "max"),
        mean_file_size_kb=("file_size_kb", "mean"),
    )
    .sort_values("mean_duration_s")
)

display(class_summary.round(2))

,file_count,mean_duration_s,median_duration_s,std_duration_s,min_duration_s,max_duration_s,mean_file_size_kb
label,,,,,,,
Defence,291,1.75,1.49,0.99,0.34,8.52,27.28
Happy,295,2.88,2.82,1.64,0.48,7.71,44.58
Warning,300,3.63,3.31,1.57,0.69,8.36,56.62
Fighting,299,3.63,3.37,1.40,0.77,7.55,56.68
Paining,287,3.82,3.63,1.75,0.72,16.82,58.61
MotherCall,296,4.14,4.18,1.37,0.57,9.01,64.65
HuntingMind,289,4.35,4.23,1.51,1.07,10.97,68.36
Angry,300,4.47,4.21,1.79,1.46,11.98,69.76
Mating,300,4.93,4.74,1.47,2.61,14.92,76.89


**What the cell above checked, and what the numbers mean:**
- *Exact duplicates* — `md5_of_file()` hashes each file's raw bytes; two files with the same MD5 are byte-identical. Result: **0**, confirming the team's cleanup (documented in `DATA_README.txt`) already removed the 8 known duplicates from this dataset.
- *Segment groups* — `group_key()` strips `" (1)"`, `" (2)"`, `"_opt"` suffixes, so files that are pieces of the same original recording collapse to one group name. **149 groups / 398 files** are segments of a longer recording cut into pieces — these must stay together in any train/test split (a group split, not a random row split), otherwise the same recording could leak across both sets.
- `class_summary` aggregates duration and file-size statistics per class with plain pandas `groupby().agg()` — no paper equivalent here, this is dataset-specific due diligence the paper doesn't report.

### Fixed-Length Window Sizing

Uses percentile statistics on `duration_s` to recommend a practical fixed recording window for a future capture pipeline.

In [14]:
duration_quantiles = df["duration_s"].quantile([0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame(name="seconds")
display(duration_quantiles.round(2))

,seconds
0.05,1.10
0.25,2.61
0.50,3.79
0.75,5.04
0.90,6.06
0.95,6.82
0.99,8.29


In [15]:
production_summary = (
    df.groupby("label")
    .agg(
        file_count=("filename", "size"),
        p05_duration_s=("duration_s", lambda s: float(s.quantile(0.05))),
        median_duration_s=("duration_s", "median"),
        p95_duration_s=("duration_s", lambda s: float(s.quantile(0.95))),
        mean_rms=("file_size_kb", "mean"),
    )
    .sort_values("median_duration_s")
)

short_share = (
    df.assign(shorter_than_2s=df["duration_s"] < 2, longer_than_6s=df["duration_s"] > 6)
    .groupby("label")[["shorter_than_2s", "longer_than_6s"]]
    .mean()
    .rename(columns={"shorter_than_2s": "pct_under_2s", "longer_than_6s": "pct_over_6s"})
    * 100
)

production_summary = production_summary.join(short_share)
display(production_summary.round(2))

,file_count,p05_duration_s,median_duration_s,p95_duration_s,mean_rms,pct_under_2s,pct_over_6s
label,,,,,,,
Defence,291,0.75,1.49,3.49,27.28,71.82,0.69
Happy,295,0.67,2.82,5.81,44.58,37.29,3.05
Warning,300,1.52,3.31,6.64,56.62,13.33,9.67
Fighting,299,1.61,3.37,5.96,56.68,12.04,4.68
Paining,287,1.38,3.63,6.70,58.61,13.24,10.10
MotherCall,296,2.13,4.18,6.43,64.65,4.39,10.47
Angry,300,2.17,4.21,7.74,69.76,3.67,16.33
HuntingMind,289,2.24,4.23,7.01,68.36,2.77,13.49
Mating,300,2.98,4.74,7.34,76.89,0.00,19.33


In [16]:
overall_p95 = float(df["duration_s"].quantile(0.95))
recommended_window_s = float(np.ceil(overall_p95 * 2) / 2)
print(f"Suggested fixed record window: about {recommended_window_s:.1f} seconds")
print("Use mono audio, a consistent sample rate, and silence trimming before feature extraction.")
print("For a later API, prefer center-cropping or loudest-window extraction so recordings of different length behave consistently.")

Suggested fixed record window: about 7.0 seconds
Use mono audio, a consistent sample rate, and silence trimming before feature extraction.
For a later API, prefer center-cropping or loudest-window extraction so recordings of different length behave consistently.


### Duration Visualizations

Interactive Plotly charts of the duration distribution, to visually check for the class-separates-by-duration shortcut.

In [17]:
def show_plotly(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

In [18]:
fig = px.box(
    df,
    x="label",
    y="duration_s",
    color="label",
    points="outliers",
    title="Duration distribution by class",
)
fig.update_layout(showlegend=False, xaxis_title="Class", yaxis_title="Duration (seconds)")
show_plotly(fig)

In [19]:
fig = px.histogram(
    df,
    x="duration_s",
    color="label",
    nbins=60,
    barmode="overlay",
    opacity=0.45,
    title="Overlaid duration histogram by class",
)
fig.update_layout(xaxis_title="Duration (seconds)", yaxis_title="Count")
show_plotly(fig)

In [20]:
fig = px.scatter(
    df,
    x="duration_s",
    y="file_size_kb",
    color="label",
    hover_data=["filename", "group"],
    title="File size versus duration",
)
fig.update_layout(xaxis_title="Duration (seconds)", yaxis_title="File size (KB)")
show_plotly(fig)

## Sample Rate Audit

`SAMPLE_RATE` above was picked once and used everywhere — this section checks that choice against the dataset's actual, original sample rates (before any resampling), and gives metrics you can rerun on any other audio folder to pick a sensible target rate for it.

### Sample Rate Metrics

Reads each file's native sample rate and summarizes the distribution, min/mode/max, and how many files the current `SAMPLE_RATE` would upsample versus downsample.

In [21]:
if "native_sr" not in df.columns:
    df["native_sr"] = [native_sample_rate(path) for path in df["path"]]

sr_counts = df["native_sr"].value_counts().sort_index()
sr_summary = pd.DataFrame({
    "file_count": sr_counts,
    "share_pct": (sr_counts / len(df) * 100).round(2),
})

display(sr_summary)

,file_count,share_pct
native_sr,,
8000,1,0.03
11025,8,0.27
22050,5,0.17
32000,24,0.81
44100,2915,98.71


In [22]:
mode_sr = int(df["native_sr"].mode().iloc[0])
min_sr = int(df["native_sr"].min())
max_sr = int(df["native_sr"].max())
pct_would_upsample = float((df["native_sr"] < SAMPLE_RATE).mean() * 100)
pct_would_downsample = float((df["native_sr"] > SAMPLE_RATE).mean() * 100)

print(f"Native sample rates present: {sorted(df['native_sr'].unique())}")
print(f"Most common native rate: {mode_sr} Hz ({(df['native_sr'] == mode_sr).mean() * 100:.1f}% of files)")
print(f"Range: {min_sr}-{max_sr} Hz")
print(f"Recommended SAMPLE_RATE ceiling for this folder: {min_sr} Hz (setting SAMPLE_RATE at or below this avoids upsampling any file)")
print(f"At the current SAMPLE_RATE ({SAMPLE_RATE} Hz): {pct_would_upsample:.1f}% of files would be upsampled, {pct_would_downsample:.1f}% downsampled")

Native sample rates present: [np.int64(8000), np.int64(11025), np.int64(22050), np.int64(32000), np.int64(44100)]
Most common native rate: 44100 Hz (98.7% of files)
Range: 8000-44100 Hz
Recommended SAMPLE_RATE ceiling for this folder: 8000 Hz (setting SAMPLE_RATE at or below this avoids upsampling any file)
At the current SAMPLE_RATE (44100 Hz): 1.3% of files would be upsampled, 0.0% downsampled


### Sample Rate Visualizations

Interactive Plotly charts of the native sample-rate distribution, by class, and against duration.

In [23]:
sr_bar_fig = px.bar(
    sr_summary.reset_index().rename(columns={"index": "native_sr"}),
    x="native_sr",
    y="file_count",
    text="share_pct",
    title="Native sample rate distribution across the dataset",
)
sr_bar_fig.update_traces(texttemplate="%{text}%", textposition="outside")
sr_bar_fig.update_layout(xaxis_title="Native sample rate (Hz)", yaxis_title="File count", xaxis_type="category")
show_plotly(sr_bar_fig)

In [24]:
sr_by_class_fig = px.histogram(
    df,
    x="native_sr",
    color="label",
    barmode="overlay",
    opacity=0.5,
    title="Native sample rate by class",
)
sr_by_class_fig.update_layout(xaxis_title="Native sample rate (Hz)", yaxis_title="Count")
show_plotly(sr_by_class_fig)

In [25]:
duration_sr_fig = px.scatter(
    df,
    x="native_sr",
    y="duration_s",
    color="label",
    hover_data=["filename", "group"],
    title="Duration versus native sample rate",
)
duration_sr_fig.update_layout(xaxis_title="Native sample rate (Hz)", yaxis_title="Duration (seconds)")
show_plotly(duration_sr_fig)

### Feature Extraction

Samples 20 files per class and computes 8 hand-crafted audio features per clip, using `SAMPLE_RATE` and `CLIP_DURATION_S` from the config above.

In [26]:
feature_samples = []
for label, frame in df.groupby("label"):
    # Sample a small, balanced slice so the feature comparison stays readable.
    sample = frame.sample(n=min(20, len(frame)), random_state=RANDOM_SEED)
    feature_samples.append(sample[["label", "filename", "path"]])

feature_sample = pd.concat(feature_samples, ignore_index=True)

In [27]:
feature_rows = []
for label, filename, path in feature_sample.itertuples(index=False, name=None):
    y, sr = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    y = fit_to_duration(y, sr, CLIP_DURATION_S)
    feature_rows.append(
        {
            "label": label,
            "filename": filename,
            "path": path,
            "duration_s": len(y) / sr,
            "rms": float(librosa.feature.rms(y=y).mean()),
            "zcr": float(librosa.feature.zero_crossing_rate(y).mean()),
            "spectral_centroid": float(librosa.feature.spectral_centroid(y=y, sr=sr).mean()),
            "spectral_bandwidth": float(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()),
            "spectral_rolloff": float(librosa.feature.spectral_rolloff(y=y, sr=sr).mean()),
            "mfcc_1": float(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=1).mean()),
            "mfcc_2": float(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=2)[1].mean()),
        }
    )

feature_df = pd.DataFrame(feature_rows)
display(feature_df.head())

,label,filename,path,duration_s,rms,zcr,spectral_centroid,spectral_bandwidth,spectral_rolloff,mfcc_1,mfcc_2
0,Angry,cat0371.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,2.0,0.045518,0.045831,1727.854763,2235.276549,2649.081862,-350.967804,167.175629
1,Angry,cat_flickr01456.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,2.0,0.054070,0.031312,1999.004374,3135.453033,3620.565390,-306.712738,146.809723
2,Angry,YashLL_Cat0289Ang.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,2.0,0.115027,0.031950,1143.461095,1590.110630,1694.277231,-311.226593,197.107208
3,Angry,Cat_Angry1035.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,2.0,0.042781,0.060767,2130.726795,2516.702048,3306.778078,-324.806732,164.172653
4,Angry,cat_flickr01305.mp3,/home/cjk/code/CevenJKnowles/Kitty3000-ML/data/raw/CatSound_originals/Angry/...,2.0,0.080677,0.040824,1287.789754,1772.190050,1982.050443,-332.327820,144.493240


**What the cell above computes, in plain terms** (a stratified sample of 20 files per class, loaded at `SAMPLE_RATE` and cropped/padded to `CLIP_DURATION_S` via `fit_to_duration`):

| Feature | What it measures |
|---|---|
| `rms` | Loudness (root-mean-square energy) |
| `zcr` | Zero-crossing rate — how "noisy" vs. tonal the signal is |
| `spectral_centroid` | Brightness — where the sound's energy is centered in frequency |
| `spectral_bandwidth` | Spread of energy around that centroid |
| `spectral_rolloff` | Frequency below which most energy sits — another brightness proxy |
| `mfcc_1`, `mfcc_2` | First two MFCCs — a compact summary of timbre, the same family of feature the paper's CNN/CDBN pipeline builds on top of |

Every `librosa.feature.*` call returns one value **per audio frame**; the trailing `.mean()` collapses that into a single number per clip, which is how a variable-length recording becomes a fixed-width row a classifier can consume. This is a much simpler feature set than the paper's: 8 hand-picked scalars per clip versus the paper's pooled deep-CNN/CDBN feature maps (GAP/FDAP pooling) — see the comparison table in "Classifier Sanity Checks" below.

### Feature Relationships

These plots test whether simple spectral descriptors already separate classes enough for later model work.

**What's plotted:** the scatter is a `px.scatter` of two raw features (spectral centroid vs. RMS) colored by class — if classes formed visibly separate clusters here, that alone would be a good sign; if they overlap heavily, it means these two features alone aren't enough (which is expected — real separation needs all 8 features together, tested properly below). The heatmap is a Pearson correlation matrix (`DataFrame.corr()`) rendered with `px.imshow`; strongly correlated feature pairs (e.g. `spectral_centroid` and `spectral_rolloff`, which both measure "brightness") tell you the features carry some redundant information rather than 8 independent signals.

In [28]:
feature_fig = px.scatter(
    feature_df,
    x="spectral_centroid",
    y="rms",
    color="label",
    hover_data=["filename", "duration_s"],
    title="Spectral centroid versus RMS on a stratified sample",
)
show_plotly(feature_fig)

In [29]:
corr_cols = [
    "duration_s",
    "rms",
    "zcr",
    "spectral_centroid",
    "spectral_bandwidth",
    "spectral_rolloff",
    "mfcc_1",
    "mfcc_2",
]
corr = feature_df[corr_cols].corr()
corr_fig = px.imshow(
    corr,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Correlation heatmap for sampled audio features",
)
show_plotly(corr_fig)

### Projection and Audio Review

This block checks whether a compact feature projection still separates classes and gives you a quick way to listen to representative clips.

**Method:** all 8 features are standardized (`StandardScaler`, zero mean / unit variance, so no single feature dominates just because of its raw scale) and then compressed to 2D — with **UMAP** if it's available (better at preserving local neighborhood structure, so visually tighter clusters), falling back to **PCA** (a simpler, deterministic linear projection) otherwise. This is purely for visualization; the actual classifiers below use all 8 features, not this 2D projection. The dropdown/slider widget below the plot lets you browse and play back individual clips per class — useful for spot-checking whether a clip's audio content matches its label.

In [30]:
if PCA is not None and StandardScaler is not None:
    embedding_source = feature_df[["duration_s", "rms", "zcr", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "mfcc_1", "mfcc_2"]]
    scaled = StandardScaler().fit_transform(embedding_source)

    # UMAP is preferable for visual inspection, but PCA is a safe fallback.
    if umap is not None:
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.2, random_state=RANDOM_SEED)
        embedding = reducer.fit_transform(scaled)
        embed_title = "UMAP projection of sampled audio features"
    else:
        reducer = PCA(n_components=2, random_state=RANDOM_SEED)
        embedding = reducer.fit_transform(scaled)
        embed_title = "PCA projection of sampled audio features"

    feature_df["embed_1"] = embedding[:, 0]
    feature_df["embed_2"] = embedding[:, 1]

    embed_fig = px.scatter(
        feature_df,
        x="embed_1",
        y="embed_2",
        color="label",
        hover_data=["filename", "duration_s", "spectral_centroid"],
        title=embed_title,
    )
    show_plotly(embed_fig)

In [31]:
# Always keep one concrete sample visible for quick manual review.
sample_row = df.sample(1, random_state=RANDOM_SEED).iloc[0]
display(sample_row[["label", "filename", "duration_s", "file_size_kb"]])
display(Audio(filename=str(sample_row["path"])))

label                 Mating
filename        cat03567.mp3
duration_s          3.736372
file_size_kb            58.2
Name: 1659, dtype: object

In [32]:
label_dropdown = widgets.Dropdown(options=classes, description="Label:")
sample_slider = widgets.IntSlider(min=0, max=0, value=0, description="Index:")
output = widgets.Output()

def refresh_slider(*_):
    """Keep the slider range aligned with the selected class."""
    subset = df[df["label"] == label_dropdown.value].reset_index(drop=True)
    sample_slider.max = max(0, len(subset) - 1)
    sample_slider.value = min(sample_slider.value, sample_slider.max)

def show_selected(*_):
    """Display the selected clip metadata and its audio player."""
    with output:
        output.clear_output(wait=True)
        subset = df[df["label"] == label_dropdown.value].reset_index(drop=True)
        row = subset.iloc[sample_slider.value]
        display(row[["label", "filename", "duration_s", "file_size_kb"]])
        display(Audio(filename=str(row["path"])))

label_dropdown.observe(refresh_slider, names="value")
sample_slider.observe(show_selected, names="value")
refresh_slider()
show_selected()
display(widgets.VBox([label_dropdown, sample_slider, output]))

print("Cross-analysis section ready.")

Cross-analysis section ready.


### Classifier Sanity Checks

These checks are not a replacement for the historical paper results. They are a lower-cost way to see whether the feature space is separable enough to support an online record-and-predict flow later.

**Pipeline, in plain terms:** `ColumnTransformer` + `StandardScaler` normalizes the 8 features; `StratifiedKFold(n_splits=5)` splits the data into 5 folds that each preserve the overall class balance; `cross_validate()` trains and tests each model on all 5 folds so every row gets used for testing exactly once. Three models are compared: an RBF-kernel `SVC`, a `RandomForestClassifier`, and a soft-voting ensemble of logistic regression + SVM.

**Comparison to the original paper (Pandeya et al. 2018):**

| Source | Features | Data | Best result |
|---|---|---|---|
| Paper, Table 2 (3x-augmented, FDAP-pooled) | Deep CNN/CDBN pooled features | Augmented dataset, full size | SVM 87.4–90.9%, Ensemble 90.8–91.1% accuracy |
| Paper, Table 3 (ensemble, by augmentation level) | Same, varying augmentation/pooling | Original → 3x-augmented | 70.7% (no augmentation, GAP) up to 91.1% (3x-aug, FDAP) |
| `.keep/EDA/cjk-audio-conv-7-EDA.ipynb` (this project, archived) | PANNs Cnn14 deep embeddings (2048-dim) | Full 2953-file originals-only set | SVM 81.8%, RF 83.7%, Ensemble 83.3% accuracy |
| **This notebook (above)** | 8 hand-crafted scalar features (rms, zcr, spectral stats, 2 MFCCs) | 20-per-class stratified sample (~200 files) | roughly 38–46% accuracy depending on `CLIP_DURATION_S` |

In [33]:
classifier_frame = feature_df.copy()
feature_columns = [
    "duration_s",
    "rms",
    "zcr",
    "spectral_centroid",
    "spectral_bandwidth",
    "spectral_rolloff",
    "mfcc_1",
    "mfcc_2",
]
X_classifier = classifier_frame[feature_columns]
y_classifier = classifier_frame["label"]

numeric_preprocess = ColumnTransformer(
    [("numeric", StandardScaler(), feature_columns)],
    remainder="drop",
)

models = {
    "svm_rbf": Pipeline([
        ("scale", numeric_preprocess),
        ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_SEED)),
    ]),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED),
    "soft_vote": Pipeline([
        ("scale", numeric_preprocess),
        ("model", VotingClassifier(
            estimators=[
                ("lr", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)),
                ("svm", SVC(kernel="rbf", probability=True, random_state=RANDOM_SEED)),
            ],
            voting="soft",
        )),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

In [34]:
for name, model in models.items():
    scores = cross_validate(
        model,
        X_classifier,
        y_classifier,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=-1,
    )
    print(f"{name}: accuracy={scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}, f1_macro={scores['test_f1_macro'].mean():.3f}")

/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packa

svm_rbf: accuracy=0.455 ± 0.081, f1_macro=0.440


random_forest: accuracy=0.430 ± 0.073, f1_macro=0.409


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


soft_vote: accuracy=0.435 ± 0.073, f1_macro=0.413


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### Local Decision Boundary

This is a quick visual check of whether the 2D projection has enough structure for a simple local classifier to separate classes.

**Method:** a `KNeighborsClassifier(n_neighbors=5)` is fit directly on the 2D UMAP/PCA coordinates (not the original 8 features), then used to predict a label for every point on a dense grid — that grid becomes the colored background regions, and the actual clips are overlaid as scatter points. This is illustrative only: it operates on a lossy 2D compression of the data and has no held-out test set, so treat it as "does this projection look locally separable at a glance," not as a measured accuracy figure. The paper has no equivalent visualization.

In [35]:
if "embed_1" in feature_df.columns and "embed_2" in feature_df.columns:
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(feature_df[["embed_1", "embed_2"]], feature_df["label"])

    x_min, x_max = feature_df["embed_1"].min() - 1, feature_df["embed_1"].max() + 1
    y_min, y_max = feature_df["embed_2"].min() - 1, feature_df["embed_2"].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

    grid_predictions = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    class_to_int = {label: index for index, label in enumerate(feature_df["label"].unique())}
    grid_codes = np.vectorize(class_to_int.get)(grid_predictions)
else:
    print("2D embedding not available yet; rerun the projection cell first.")

In [36]:
if "embed_1" in feature_df.columns and "embed_2" in feature_df.columns:
    boundary_fig = go.Figure(
        data=go.Heatmap(
            x=xx[0],
            y=yy[:, 0],
            z=grid_codes,
            colorscale="Viridis",
            opacity=0.35,
            showscale=False,
            hoverinfo="skip",
        )
    )
    for label in feature_df["label"].unique():
        subset = feature_df[feature_df["label"] == label]
        boundary_fig.add_trace(
            go.Scatter(
                x=subset["embed_1"],
                y=subset["embed_2"],
                mode="markers",
                name=label,
                marker=dict(size=7, opacity=0.8),
            )
        )
    boundary_fig.update_layout(
        title="KNN decision boundary on the 2D feature projection",
        xaxis_title="embed_1",
        yaxis_title="embed_2",
    )
    show_plotly(boundary_fig)

## PANN Embeddings (2-second clips)

This section extracts deep audio embeddings using a pretrained PANNs `Cnn14` model (`panns_inference`, checkpoint `Cnn14_mAP=0.431.pth`, run on CPU/`cpu`) for every file in the `df` dataframe already loaded above — the same 2953-file inventory used throughout this notebook, not a fresh reload.

Unlike the hand-crafted-feature sections above, PANNs was trained at **32000 Hz**, not this notebook's `SAMPLE_RATE = 44100` — so audio is reloaded at 32000 Hz here specifically for the model. Each clip is still cropped/padded to this notebook's `CLIP_DURATION_S = 2.0` seconds via the shared `fit_to_duration()` helper, so results below are directly comparable to this notebook's own 2-second hand-crafted-feature results above, and to the archived **full-length** (uncropped) PANNs baseline in `.keep/EDA/cjk-audio-conv-7-EDA.ipynb` (81.8% / 83.7% / 83.3% accuracy for SVM / RF / Ensemble).

In [37]:
from panns_inference import AudioTagging
from sklearn.preprocessing import LabelEncoder

PANN_SAMPLE_RATE = 32000  # Hz — PANNs Cnn14 was trained at this rate, independent of this notebook's SAMPLE_RATE

pann_model = AudioTagging(checkpoint_path=None, device="cpu")

Checkpoint path: /home/cjk/panns_data/Cnn14_mAP=0.431.pth


GPU number: 1


In [38]:
pann_embeddings = []
pann_labels = []

for label, path in df[["label", "path"]].itertuples(index=False, name=None):
    audio, sr = librosa.load(str(path), sr=PANN_SAMPLE_RATE, mono=True)
    audio = fit_to_duration(audio, sr, CLIP_DURATION_S)
    _, embedding = pann_model.inference(audio[None, :])
    pann_embeddings.append(embedding[0])
    pann_labels.append(label)

X_pann = np.vstack(pann_embeddings)
y_pann = np.array(pann_labels)
print(X_pann.shape, y_pann.shape)

Note: Illegal Audio-MPEG-Header 0xbf082800 at offset 7536.
Note: Trying to resync...
Note: Hit end of (available) data during resync.


(2953, 2048) (2953,)


### Dimensionality Reduction

Same three projections as the archived baseline (PCA 2D, PCA 3D, UMAP), rendered as interactive Plotly charts instead of static matplotlib.

In [39]:
pann_pca_2d = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_pann)
pann_pca_2d_df = pd.DataFrame(pann_pca_2d, columns=["pc1", "pc2"])
pann_pca_2d_df["label"] = y_pann

pann_pca_2d_fig = px.scatter(
    pann_pca_2d_df, x="pc1", y="pc2", color="label",
    title="PANNs embeddings (2s clips) — PCA to 2D",
)
show_plotly(pann_pca_2d_fig)

In [40]:
pann_pca_3d = PCA(n_components=3, random_state=RANDOM_SEED).fit_transform(X_pann)
pann_pca_3d_df = pd.DataFrame(pann_pca_3d, columns=["pc1", "pc2", "pc3"])
pann_pca_3d_df["label"] = y_pann

pann_pca_3d_fig = px.scatter_3d(
    pann_pca_3d_df, x="pc1", y="pc2", z="pc3", color="label",
    title="PANNs embeddings (2s clips) — PCA to 3D",
)
show_plotly(pann_pca_3d_fig)

In [41]:
pann_umap_reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED)
pann_umap = pann_umap_reducer.fit_transform(X_pann)
pann_umap_df = pd.DataFrame(pann_umap, columns=["umap_1", "umap_2"])
pann_umap_df["label"] = y_pann

pann_umap_fig = px.scatter(
    pann_umap_df, x="umap_1", y="umap_2", color="label",
    title="PANNs embeddings (2s clips) — UMAP to 2D",
)
show_plotly(pann_umap_fig)

### Classifier Benchmarks

Same evaluation as the archived baseline: 10-fold stratified cross-validation directly on the raw 2048-dim embeddings (zero-variance dimensions dropped first, for free — no information lost).

In [42]:
X_pann_live = X_pann[:, X_pann.var(axis=0) > 0]
pann_label_encoder = LabelEncoder()
y_pann_enc = pann_label_encoder.fit_transform(y_pann)

pann_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

pann_models = {
    "svm_rbf": SVC(kernel="rbf", probability=True, random_state=RANDOM_SEED),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    "soft_vote": VotingClassifier(
        estimators=[
            ("svm", SVC(kernel="rbf", probability=True, random_state=RANDOM_SEED)),
            ("rf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)),
        ],
        voting="soft",
    ),
}

pann_results = []
for name, model in pann_models.items():
    scores = cross_validate(
        model, X_pann_live, y_pann_enc, cv=pann_cv,
        scoring=["accuracy", "f1_macro", "roc_auc_ovr"],
    )
    pann_results.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro": scores["test_f1_macro"].mean(),
        "roc_auc_ovr": scores["test_roc_auc_ovr"].mean(),
    })
    print(f"{name}: accuracy={scores['test_accuracy'].mean():.4f} (±{scores['test_accuracy'].std():.4f}), f1_macro={scores['test_f1_macro'].mean():.4f}, AUC={scores['test_roc_auc_ovr'].mean():.4f}")

pann_results_df = pd.DataFrame(pann_results)
display(pann_results_df.round(4))

/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/cjk/code/CevenJKnowles/Kitty3000-ML/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


soft_vote: accuracy=0.7755 (±0.0283), f1_macro=0.7733, AUC=0.9689


,model,accuracy,accuracy_std,f1_macro,roc_auc_ovr
0,svm_rbf,0.7646,0.0263,0.7614,0.9693
1,random_forest,0.7667,0.0324,0.7641,0.9636
2,soft_vote,0.7755,0.0283,0.7733,0.9689


In [43]:
comparison_df = pd.DataFrame([
    {"feature_type": "Hand-crafted (this notebook, 2s)", "model": "svm_rbf", "accuracy": 0.455},
    {"feature_type": "Hand-crafted (this notebook, 2s)", "model": "random_forest", "accuracy": 0.430},
    {"feature_type": "Hand-crafted (this notebook, 2s)", "model": "soft_vote", "accuracy": 0.435},
    {"feature_type": "PANNs embeddings (this notebook, 2s crop)", "model": "svm_rbf", "accuracy": pann_results_df.set_index("model").loc["svm_rbf", "accuracy"]},
    {"feature_type": "PANNs embeddings (this notebook, 2s crop)", "model": "random_forest", "accuracy": pann_results_df.set_index("model").loc["random_forest", "accuracy"]},
    {"feature_type": "PANNs embeddings (this notebook, 2s crop)", "model": "soft_vote", "accuracy": pann_results_df.set_index("model").loc["soft_vote", "accuracy"]},
    {"feature_type": "Archived PANNs (full-length, no crop)", "model": "svm_rbf", "accuracy": 0.8175},
    {"feature_type": "Archived PANNs (full-length, no crop)", "model": "random_forest", "accuracy": 0.8374},
    {"feature_type": "Archived PANNs (full-length, no crop)", "model": "soft_vote", "accuracy": 0.8331},
])

comparison_fig = px.bar(
    comparison_df, x="model", y="accuracy", color="feature_type", barmode="group",
    title="Accuracy: hand-crafted features vs. PANNs (2s crop) vs. archived PANNs (full length)",
)
comparison_fig.update_layout(yaxis_tickformat=".0%", yaxis_title="Accuracy")
show_plotly(comparison_fig)

### Local Decision Boundary (PANNs UMAP)

Same illustrative technique as the hand-crafted-feature section above, applied to the PANNs UMAP projection instead — a `KNeighborsClassifier(n_neighbors=5)` fit on the 2D coordinates, used only to shade a decision-boundary background behind the actual points.

In [44]:
pann_knn = KNeighborsClassifier(n_neighbors=5)
pann_knn.fit(pann_umap_df[["umap_1", "umap_2"]], pann_umap_df["label"])

x_min, x_max = pann_umap_df["umap_1"].min() - 1, pann_umap_df["umap_1"].max() + 1
y_min, y_max = pann_umap_df["umap_2"].min() - 1, pann_umap_df["umap_2"].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

grid_predictions = pann_knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
pann_class_to_int = {label: i for i, label in enumerate(pann_umap_df["label"].unique())}
grid_codes = np.vectorize(pann_class_to_int.get)(grid_predictions)

pann_boundary_fig = go.Figure(
    data=go.Heatmap(
        x=xx[0], y=yy[:, 0], z=grid_codes,
        colorscale="Viridis", opacity=0.35, showscale=False, hoverinfo="skip",
    )
)
for label in pann_umap_df["label"].unique():
    subset = pann_umap_df[pann_umap_df["label"] == label]
    pann_boundary_fig.add_trace(
        go.Scatter(x=subset["umap_1"], y=subset["umap_2"], mode="markers", name=label, marker=dict(size=5, opacity=0.7))
    )
pann_boundary_fig.update_layout(
    title="KNN decision boundary on PANNs UMAP projection (2s clips)",
    xaxis_title="umap_1", yaxis_title="umap_2",
)
show_plotly(pann_boundary_fig)